In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from econml.dml import LinearDML
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

# ==========================================
# 1. 读取数据
# ==========================================
file_path = r"【0405】左连结数据整合 - 副本.xlsx"
df = pd.read_excel(file_path, sheet_name="Sheet1")

In [ ]:
# ==========================================
# 2. 面板数据缺失值处理（按地级市分组插值）
# ==========================================
# 提取需要插值的列：诉求量(被解释变量) 和 R列到AH列(控制变量基础数据)
# 这里以列名或列索引来提取，假设17:34对应R到AJ列
target_cols = ['诉求量'] + df.columns[17:36].tolist()
# 按照地级市分组，使用线性插值处理时间序列上的缺失值
# limit_direction='both' 确保时间序列头尾的缺失值也能向前或向后填充
df[target_cols] = df.groupby('地级市')[target_cols].apply(
    lambda x: x.interpolate(method='linear', limit_direction='both')
).reset_index(level=0, drop=True)

#df.to_excel("1_插值补充后数据.xlsx", index=False)

In [ ]:
# ==========================================
# 3. 构造文献要求的控制变量
# ==========================================
# 注意：等号右侧的 ['列名'] 需要你对照生成的 "1_插值补充后数据.xlsx" 里的实际基础表头进行替换
df['人力资本水平'] = df['普通高等学校在校学生数(人)'] / df['户籍人口(万人)']
df['传统基础设施'] = np.log(df['公路货运量(万吨)'] + 1) # 加1防止出现log(0)
df['互联网发展水平'] = np.log(df['电信业务收入(万元)'] + 1)
df['金融发展水平'] = df['年末金融机构各项贷款余额(万元)'] / df['地区生产总值(万元)']
df['财政压力水平'] = df['地方财政一般预算内支出(万元)'] / df['地方财政一般预算内收入(万元)']
df['科学支出水平'] = np.log(df['科学支出(万元)'] + 1)
df['城市发展水平'] = np.log(df['人均地区生产总值(元)'])
df['外商投资水平'] = np.log(df['外商投资企业数(个)'])
df['交通便捷程度'] = df['高速公路里程(公里)'] / df['户籍人口(万人)']
# 将构造好的控制变量放入列表
controls = ['地区生产总值(万元)','人口密度(人／平方公里)', '第三产业增加值占GDP比重(%)', '人力资本水平', '每百人公共图书馆藏书(册、件)',
            '传统基础设施', '互联网发展水平', '金融发展水平', '财政压力水平', '科学支出水平','城市发展水平','外商投资水平','生活垃圾无害化处理率(%)','交通便捷程度']

# 剔除在构造变量后仍然存在缺失值（如缺乏某城市整体面积数据导致无法插值）的样本
df.dropna(subset=['诉求量', 'post（开放数据平台时间虚拟变量）'] + controls, inplace=True)
#df.to_excel("2_处理后含控制变量数据.xlsx", index=False)
